# LeetCode 3171 — Find Subarray With Bitwise OR Closest to K bằng Python

- Link: https://leetcode.com/problems/find-subarray-with-bitwise-or-closest-to-k/
- Độ khó: **Hard**
- Chủ đề: **Tính đơn điệu của OR** — với đầu phải cố định, OR chỉ tăng (bật thêm bit) khi kéo dài sang trái, nhận `≤ 31` giá trị khác nhau; kỹ thuật tập giá trị OR, ghi đè tại chỗ (LC 1521), cửa sổ trượt đếm bit

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minimumDifference(self, nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Với `r` cố định, `OR(l..r)` **không giảm** khi `l` giảm (thêm phần tử chỉ bật thêm bit) và chỉ nhận `≤ 31` giá trị khác nhau.
2. Duy trì tập các giá trị OR của mảng con kết thúc tại `r` ⇒ `O(n log V)`.
3. Mẹo ghi đè tại chỗ: `nums[i] |= nums[r]` lùi về trái, dừng khi không đổi — chỉ các giá trị **mới** cần so với `k`.
4. Cửa sổ trượt: OR của cửa sổ đơn điệu theo bao hàm ⇒ hai con trỏ quanh ngưỡng `k` (cần bộ đếm bit để "bỏ" phần tử).
5. Kiểm chứng bằng brute `O(n²)`.

## 2. Đề bài & ví dụ

Cho mảng `nums` và số nguyên `k`. Tìm mảng con (liên tiếp, khác rỗng) `nums[l..r]` sao cho `|k − (nums[l] OR … OR nums[r])|` nhỏ nhất; trả về giá trị nhỏ nhất đó.

**Ví dụ 1:** `nums = [1,2,4,5], k = 3` → `0`. **Ví dụ 2:** `nums = [1,3,1,3], k = 2` → `1`. **Ví dụ 3:** `nums = [1], k = 10` → `9`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`; `1 ≤ nums[i] ≤ 10^9`; `1 ≤ k ≤ 10^9`.

In [1]:
from typing import List
import random, time

examples = [([1, 2, 4, 5], 3, 0), ([1, 3, 1, 3], 2, 1), ([1], 10, 9)]
for a, k, exp in examples:
    print(a, 'k =', k, '-> kỳ vọng', exp)

[1, 2, 4, 5] k = 3 -> kỳ vọng 0
[1, 3, 1, 3] k = 2 -> kỳ vọng 1
[1] k = 10 -> kỳ vọng 9


## 3. Kiến thức nền

### 3.1. OR chỉ bật thêm bit

`x | y ⊇ x` ⇒ `x | y ≥ x`. Với `r` cố định, `OR(r..r) ≤ OR(r−1..r) ≤ …`, và mỗi lần tăng thì bật thêm ít nhất một bit ⇒ `≤ 30` lần tăng (giá trị `< 2^30`).

In [2]:
def or_values_ending_at(nums, r):
    out = []; cur = 0
    for l in range(r, -1, -1):
        cur |= nums[l]; out.append(cur)
    return out

rng = random.Random(3171)
a = [1 << rng.randint(0, 9) for _ in range(14)]
v = or_values_ending_at(a, 13)
print('OR(l..13) khi l giảm:', v)
print('khác nhau:', len(set(v)), '| không giảm:', all(v[i] <= v[i + 1] for i in range(len(v) - 1)))

OR(l..13) khi l giảm: [32, 288, 292, 292, 420, 420, 932, 996, 996, 1012, 1012, 1012, 1013, 1013]
khác nhau: 8 | không giảm: True


### 3.2. Ngưỡng đơn điệu

Vì `OR(l..r)` không giảm khi `l` giảm, các `l` có `OR(l..r) ≤ k` lập thành đoạn `[L(r), r]` (có thể rỗng). Ứng viên gần `k` nhất là `OR(L(r)..r)` (lớn nhất mà `≤ k`) và `OR(L(r)−1..r)` (nhỏ nhất mà `> k`). Khi `r` tăng, `L(r)` không giảm ⇒ hai con trỏ.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đây là bài "đối ngẫu" của LC 1521 (AND gần target) và LC 3209 (đếm AND = k): thay AND bằng OR. Vì mỗi đầu phải chỉ có ≤ 31 giá trị OR khác nhau, ta thử **tất cả** chúng — tổng `O(31 n)` so sánh.

Ví dụ 2 (`[1, 3, 1, 3]`, `k = 2`): mọi OR đều là `1` hoặc `3` ⇒ `|2 − 1| = |2 − 3| = 1`.

## 5. Cách 1 — Brute Force `O(n²)`

In [3]:
def brute(nums, k):
    best = float('inf'); n = len(nums)
    for l in range(n):
        cur = 0
        for r in range(l, n):
            cur |= nums[r]
            best = min(best, abs(cur - k))
    return best

for a, k, exp in examples:
    print('brute:', brute(a, k), '| kỳ vọng', exp)

brute: 0 | kỳ vọng 0
brute: 1 | kỳ vọng 1
brute: 9 | kỳ vọng 9


Tối ưu nhỏ hợp lệ: khi `cur > k` (sau khi đã ghi nhận) thì kéo dài thêm chỉ làm `cur` lớn hơn ⇒ `break`. Vẫn `O(n²)` xấu nhất (khi mọi OR `≤ k`).

## 6. Ý tưởng thuật toán / bước đột phá

```
ans = ∞
for r, x in enumerate(nums):
    ans = min(ans, |x − k|)
    i = r − 1
    while i ≥ 0 and nums[i] | x != nums[i]:      # nums[i] đang lưu OR(i..r−1)
        nums[i] |= x                             # thành OR(i..r)
        ans = min(ans, |nums[i] − k|)
        i −= 1
```

Các ô không đổi đã được so trước đó (cùng giá trị) ⇒ không cần so lại.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tập giá trị OR kết thúc tại `r`

In [4]:
def or_set(nums, k):
    best = float('inf'); cur = set()
    for x in nums:
        cur = {v | x for v in cur}; cur.add(x)
        for v in cur:
            d = abs(v - k)
            if d < best:
                best = d
    return best

for a, k, exp in examples:
    print('tập OR:', or_set(a, k), '| kỳ vọng', exp)

tập OR: 0 | kỳ vọng 0
tập OR: 1 | kỳ vọng 1
tập OR: 9 | kỳ vọng 9


### 7.2. Cách 3 — Ghi đè tại chỗ (lời giải chính)

In [5]:
def inplace(nums, k):
    b = list(nums); best = float('inf')
    for r, x in enumerate(b):
        best = min(best, abs(x - k))
        i = r - 1
        while i >= 0 and b[i] | x != b[i]:
            b[i] |= x
            best = min(best, abs(b[i] - k))
            i -= 1
    return best

for a, k, exp in examples:
    print('ghi đè tại chỗ:', inplace(a, k), '| kỳ vọng', exp)

ghi đè tại chỗ: 0 | kỳ vọng 0
ghi đè tại chỗ: 1 | kỳ vọng 1
ghi đè tại chỗ: 9 | kỳ vọng 9


### 7.3. Cách 4 — Cửa sổ trượt với bộ đếm bit

Giữ cửa sổ `[l, r]` có OR `≤ k` khi có thể: thêm `nums[r]`; trong khi OR `> k`, ghi nhận `OR − k` rồi bỏ `nums[l]` (giảm bộ đếm bit, bit nào về 0 thì tắt). Sau đó, nếu cửa sổ khác rỗng, ghi nhận `k − OR`.

In [6]:
def sliding(nums, k):
    B = 31; cnt = [0] * B; cur = 0; l = 0; best = float('inf')
    for r, x in enumerate(nums):
        for b in range(B):
            if x >> b & 1:
                cnt[b] += 1
        cur |= x
        while l <= r and cur > k:
            best = min(best, cur - k)
            y = nums[l]
            for b in range(B):
                if y >> b & 1:
                    cnt[b] -= 1
                    if cnt[b] == 0:
                        cur &= ~(1 << b)
            l += 1
        if l <= r:
            best = min(best, k - cur)
    return best

for a, k, exp in examples:
    print('cửa sổ trượt:', sliding(a, k), '| kỳ vọng', exp)

cửa sổ trượt: 0 | kỳ vọng 0
cửa sổ trượt: 1 | kỳ vọng 1
cửa sổ trượt: 9 | kỳ vọng 9


Vì sao đủ: với mỗi `r`, cửa sổ cuối `[l, r]` là `[L(r), r]` (OR lớn nhất `≤ k`), và `OR(L(r)−1..r) > k` đã được ghi nhận khi bỏ `nums[L(r)−1]` — ở bước `r` hoặc ở bước trước với cửa sổ con của nó (OR nhỏ hơn hoặc bằng, nên gần `k` hơn hoặc bằng).

### 7.4. Cách 5 — Sparse table + tìm nhị phân

OR idempotent ⇒ sparse table trả lời `OR(l..r)` `O(1)`. Với mỗi `r`, tìm nhị phân `L(r)` và xét hai ứng viên quanh ngưỡng.

In [7]:
def sparse(nums, k):
    n = len(nums); sp = [list(nums)]; j = 1
    while (1 << j) <= n:
        prev = sp[-1]; h = 1 << (j - 1)
        sp.append([prev[i] | prev[i + h] for i in range(n - (1 << j) + 1)]); j += 1
    def q(l, r):
        t = (r - l + 1).bit_length() - 1
        return sp[t][l] | sp[t][r - (1 << t) + 1]
    best = float('inf')
    for r in range(n):
        lo, hi = 0, r + 1                       # l nhỏ nhất với OR(l..r) <= k (r+1 nếu không có)
        while lo < hi:
            mid = (lo + hi) // 2
            if q(mid, r) <= k:
                hi = mid
            else:
                lo = mid + 1
        if lo <= r:
            best = min(best, k - q(lo, r))
        if lo >= 1:
            best = min(best, q(lo - 1, r) - k)
    return best

for a, k, exp in examples:
    print('sparse table:', sparse(a, k), '| kỳ vọng', exp)

sparse table: 0 | kỳ vọng 0
sparse table: 1 | kỳ vọng 1
sparse table: 9 | kỳ vọng 9


## 8. TRACE từng bước

Trace cách 3 trên `nums = [4, 1, 2, 8, 1]`, `k = 10`.

In [8]:
def trace(nums, k):
    b = list(nums); best = float('inf')
    for r, x in enumerate(b):
        best = min(best, abs(x - k)); new_vals = [x]
        i = r - 1
        while i >= 0 and b[i] | x != b[i]:
            b[i] |= x; best = min(best, abs(b[i] - k)); new_vals.append(b[i]); i -= 1
        print(f'r = {r} (x = {x}): b[0..r] = {b[:r + 1]} | giá trị mới so với k: {new_vals} | best = {best}')
    print('đáp án:', best, '| brute:', brute(nums, k))

trace([4, 1, 2, 8, 1], 10)

r = 0 (x = 4): b[0..r] = [4] | giá trị mới so với k: [4] | best = 6
r = 1 (x = 1): b[0..r] = [5, 1] | giá trị mới so với k: [1, 5] | best = 5
r = 2 (x = 2): b[0..r] = [7, 3, 2] | giá trị mới so với k: [2, 3, 7] | best = 3
r = 3 (x = 8): b[0..r] = [15, 11, 10, 8] | giá trị mới so với k: [8, 10, 11, 15] | best = 0
r = 4 (x = 1): b[0..r] = [15, 11, 11, 9, 1] | giá trị mới so với k: [1, 9, 11] | best = 0
đáp án: 0 | brute: 0


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def minimumDifference(self, nums: List[int], k: int) -> int:
        b = list(nums)
        best = abs(b[0] - k)
        for r, x in enumerate(b):
            d = abs(x - k)
            if d < best:
                best = d
            i = r - 1
            while i >= 0 and b[i] | x != b[i]:
                b[i] |= x
                d = abs(b[i] - k)
                if d < best:
                    best = d
                i -= 1
            if best == 0:
                return 0
        return best

for a, k, exp in examples:
    got = Solution().minimumDifference(a, k)
    print(got, 'OK' if got == exp else 'SAI')

0 OK
1 OK
9 OK


## 10. Vì sao thuật toán đúng

1. **Bất biến:** trước bước `r`, `b[i] = OR(i..r−1)` với `i < r`. Gán `b[i] |= x` cho `OR(i..r)`. Nếu `b[i] | x = b[i]` (tức `x ⊆ b[i]`), thì với `i' < i`, `b[i'] ⊇ b[i] ⊇ x` ⇒ cũng không đổi ⇒ dừng sớm vẫn giữ bất biến.
2. **Đủ ứng viên:** mỗi giá trị `OR(i..r)` hoặc vừa được tính (đã so với `k`), hoặc bằng `OR(i..r−1)` — đã được so ở bước trước (hoặc sớm hơn nữa) ⇒ mọi giá trị OR của mọi mảng con đều đã được so.
3. **Khấu hao:** mỗi lần ghi `b[i]` bật thêm ít nhất 1 bit ⇒ `≤ 30` lần mỗi ô ⇒ `O(30 n)`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(n²)` | `O(1)` |
| 2. Tập OR | `O(n log V)` | `O(log V)` |
| 3. Ghi đè tại chỗ | `O(n log V)` | `O(n)` (hoặc `O(1)` nếu sửa `nums`) |
| 4. Cửa sổ trượt bit | `O(n log V)` | `O(log V)` |
| 5. Sparse table | `O(n log n)` | `O(n log n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'or_set': or_set, 'inplace': inplace, 'sliding': sliding, 'sparse': sparse,
         'submit': lambda a, k: Solution().minimumDifference(a, k)}
edge = [([5], 5, 0), ([5], 1, 4), ([8, 8], 1, 7), ([1, 2], 4, 1), ([6, 1], 5, 1),
        ([1, 2, 4, 8], 15, 0), ([10 ** 9], 1, 10 ** 9 - 1)]
for a, k, exp in examples + edge:
    assert brute(a, k) == exp, (a, k, brute(a, k))
    for name, f in impls.items():
        assert f(a, k) == exp, (name, a, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `[1, 2]`, `k = 4`: OR có thể là `1, 2, 3` ⇒ gần nhất `3`, hiệu `1`. `[6, 1]`, `k = 5`: `6` (hiệu 1), `1` (hiệu 4), `7` (hiệu 2) ⇒ `1`.

### 12.2. Đối chiếu ngẫu nhiên với brute

In [11]:
cnt = 0
for _ in range(500):
    n = rng.randint(1, 25); mask = rng.choice([7, 63, 2 ** 20 - 1])
    a = [rng.randint(1, mask) for _ in range(n)]
    k = rng.choice([rng.randint(1, mask), a[0], 1])
    exp = brute(a, k)
    for name, f in impls.items():
        assert f(a, k) == exp, (name, a, k)
    cnt += 1
print('Khớp brute trên', cnt, 'bộ ngẫu nhiên.')

Khớp brute trên 500 bộ ngẫu nhiên.


### 12.3. Ca lớn `n = 10^5`

In [12]:
N = 10 ** 5
big = {
    'ngẫu nhiên': ([rng.randint(1, 10 ** 9) for _ in range(N)], rng.randint(1, 10 ** 9)),
    'lũy thừa 2 (nhiều giá trị OR)': ([1 << rng.randint(0, 29) for _ in range(N)], 123456789),
    'toàn 1, k lớn': ([1] * N, 10 ** 9),
}
for name, (a, k) in big.items():
    t = time.time(); x = Solution().minimumDifference(a, k); t1 = time.time() - t
    t = time.time(); y = or_set(a, k); t2 = time.time() - t
    assert x == y
    print(f'{name:30s}: {x:10d} | nộp {t1:.2f}s | tập OR {t2:.2f}s')
assert Solution().minimumDifference([1] * N, 10 ** 9) == 10 ** 9 - 1
print('Toàn 1: mọi OR = 1 ⇒ k − 1: đúng.')

ngẫu nhiên                    :       2362 | nộp 0.09s | tập OR 0.12s


lũy thừa 2 (nhiều giá trị OR) :       7446 | nộp 0.49s | tập OR 0.51s
toàn 1, k lớn                 :  999999999 | nộp 0.01s | tập OR 0.02s
Toàn 1: mọi OR = 1 ⇒ k − 1: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét các OR `≤ k`

In [13]:
def wrong_only_below(nums, k):
    best = float('inf'); cur = set()
    for x in nums:
        cur = {v | x for v in cur}; cur.add(x)
        for v in cur:
            if v <= k:
                best = min(best, k - v)                              # SAI
    return best

args = ([1, 4], 3)
print('sai :', wrong_only_below(*args), '| đúng:', Solution().minimumDifference(*args))
args = ([8, 8], 1)
print('sai :', wrong_only_below(*args), '| đúng:', Solution().minimumDifference(*args))

sai : 2 | đúng: 1
sai : inf | đúng: 7


**Vì sao sai:** giá trị gần nhất có thể **lớn hơn** `k`; nếu mọi phần tử `> k` thì không có ứng viên nào (trả `inf`).

### Lỗi 2 — Brute "dừng sớm" trước khi ghi nhận OR vượt `k`

In [14]:
def wrong_break_before(nums, k):
    best = float('inf'); n = len(nums)
    for l in range(n):
        cur = 0
        for r in range(l, n):
            if cur | nums[r] > k:
                break                                                    # SAI: chưa so OR vượt k
            cur |= nums[r]
            best = min(best, k - cur)
    return best

args = ([1, 4], 3)
print('sai :', wrong_break_before(*args), '| đúng:', Solution().minimumDifference(*args))

sai : 2 | đúng: 1


**Vì sao sai:** `4` (hiệu 1) gần `3` hơn `1` (hiệu 2) nhưng bị bỏ qua. Được `break` **sau** khi ghi nhận giá trị đầu tiên vượt `k`.

### Lỗi 3 — Chỉ xét từng phần tử và cả mảng

In [15]:
def wrong_singles_and_all(nums, k):
    tot = 0
    for x in nums:
        tot |= x
    return min(min(abs(x - k) for x in nums), abs(tot - k))           # SAI

print('sai :', wrong_singles_and_all([1, 2, 4], 3), '| đúng:', Solution().minimumDifference([1, 2, 4], 3))

sai : 1 | đúng: 0


**Vì sao sai:** mảng con ở giữa (`[1, 2]` có OR 3) có thể tốt nhất.

### Lỗi 4 — Cửa sổ trượt "bỏ" phần tử bằng XOR/trừ thay vì bộ đếm bit

In [16]:
def wrong_remove_xor(nums, k):
    cur = 0; l = 0; best = float('inf')
    for r, x in enumerate(nums):
        cur |= x
        while l <= r and cur > k:
            best = min(best, cur - k)
            cur ^= nums[l]                                               # SAI: bit có thể còn trong phần tử khác
            l += 1
        if l <= r:
            best = min(best, k - cur)
    return best

args = ([1, 3], 2)
print('sai :', wrong_remove_xor(*args), '| đúng:', Solution().minimumDifference(*args))

sai : 0 | đúng: 1


**Vì sao sai:** OR không có phép "trừ": bỏ `1` khỏi cửa sổ `[1, 3]` bằng XOR tắt bit 0 dù `3` vẫn còn bit đó ⇒ OR "còn lại" thành `2` thay vì `3` và báo hiệu `0` — một giá trị OR không tồn tại. Phải đếm số phần tử bật từng bit.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Mảng con + OR/AND/GCD ⇒ "≤ log V giá trị khác nhau kết thúc tại `r`".
- "Gần `k` nhất" với hàm đơn điệu ⇒ xét hai ứng viên quanh ngưỡng.
- Cửa sổ trượt với OR/AND ⇒ cần **bộ đếm bit** để xoá phần tử.
- Có thể thoát sớm khi đạt hiệu 0.

## 15. Biến thể của bài

1. **LC 1521:** AND gần `target` nhất — đối ngẫu.
2. **LC 898:** đếm số giá trị OR **khác nhau** của mọi mảng con ⇒ hợp các tập.
3. **LC 2411:** với mỗi `l`, mảng con ngắn nhất bắt đầu tại `l` có OR lớn nhất.
4. Gần `k` nhất với ràng buộc độ dài mảng con `≥ L` ⇒ cửa sổ/sparse table.

In [17]:
# Biến thể 2 (LC 898): số giá trị OR khác nhau
def distinct_ors(nums):
    allv = set(); cur = set()
    for x in nums:
        cur = {v | x for v in cur}; cur.add(x); allv |= cur
    return len(allv)

def distinct_ors_brute(nums):
    s = set(); n = len(nums)
    for l in range(n):
        c = 0
        for r in range(l, n):
            c |= nums[r]; s.add(c)
    return len(s)

for _ in range(200):
    a = [rng.randint(0, 31) for _ in range(rng.randint(1, 15))]
    assert distinct_ors(a) == distinct_ors_brute(a)
assert distinct_ors([1, 1, 2]) == 3 and distinct_ors([1, 2, 4]) == 6
print('Biến thể LC 898: khớp brute; [1,1,2] -> 3, [1,2,4] -> 6.')

Biến thể LC 898: khớp brute; [1,1,2] -> 3, [1,2,4] -> 6.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Brute | `O(n²)` | Oracle |
| 2 | Tập OR kết thúc tại `r` | `O(n log V)` | Tổng quát |
| 3 | Ghi đè tại chỗ | `O(n log V)` | **Nộp bài** |
| 4 | Cửa sổ trượt bit | `O(n log V)` | Bộ nhớ nhỏ |
| 5 | Sparse table + nhị phân | `O(n log n)` | Truy vấn OR đoạn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [2, 4, 8]`, `k = 5`.

**Bài 2 (tính tay):** `nums = [5, 2, 9]`, `k = 14` — liệt kê mọi giá trị OR.

**Bài 3 (điền TODO):** trả về luôn một mảng con `(l, r)` **ngắn nhất** đạt hiệu nhỏ nhất.

**Bài 4 (tự viết lại):** phiên bản AND (LC 1521) — `|AND − k|` nhỏ nhất.

**Bài 5 (mini challenge):** trả lời `q` truy vấn `k` khác nhau (offline) trên cùng một mảng trong `O(n log V + q log(n log V))`.

In [18]:
def shortest_best_todo(nums, k):
    # TODO
    raise NotImplementedError

def and_closest_todo(nums, k):
    # TODO
    raise NotImplementedError

def many_k_todo(nums, ks):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Các OR: `2, 4, 8, 6, 12, 14`. (2) Gần 5 nhất: `4` hoặc `6`. (3) `1`.

**Bài 2.** (1) `5, 2, 9, 7, 11, 15`. (2) Gần 14: `15` (hiệu 1). (3) `1`.

**Bài 3.** (1) Với mỗi `r`, lưu cặp `(giá trị OR, l lớn nhất)` — `l` lớn nhất cho mảng con ngắn nhất với giá trị đó. (2) So `(hiệu, độ dài)`. (3) Kiểm bằng brute.

**Bài 4.** (1) Thay `|` bằng `&`. (2) Ghi đè `b[i] &= x` khi `b[i] & x != b[i]`. (3) Kiểm brute.

**Bài 5.** (1) Thu mọi giá trị OR khác nhau (≤ `31 n`). (2) Sắp xếp. (3) Mỗi truy vấn `bisect` lấy hai láng giềng.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 4
assert Solution().minimumDifference([2, 4, 8], 5) == 1
assert Solution().minimumDifference([5, 2, 9], 14) == 1

def and_closest(nums, k):
    b = list(nums); best = float('inf')
    for r, x in enumerate(b):
        best = min(best, abs(x - k)); i = r - 1
        while i >= 0 and b[i] & x != b[i]:
            b[i] &= x; best = min(best, abs(b[i] - k)); i -= 1
    return best

def and_brute(nums, k):
    best = float('inf')
    for l in range(len(nums)):
        c = -1
        for r in range(l, len(nums)):
            c &= nums[r]; best = min(best, abs(c - k))
    return best

for _ in range(300):
    a = [rng.randint(1, 63) for _ in range(rng.randint(1, 15))]; k = rng.randint(0, 70)
    assert and_closest(a, k) == and_brute(a, k)
assert and_closest([9, 12, 3, 7, 15], 5) == 2          # ví dụ LC 1521
print('Bài 1: 1 | Bài 2: 1 | Bài 4 (AND, LC 1521): khớp brute và ví dụ chuẩn.')

Bài 1: 1 | Bài 2: 1 | Bài 4 (AND, LC 1521): khớp brute và ví dụ chuẩn.


In [20]:
# Bài 3, 5
from bisect import bisect_left
def shortest_best(nums, k):
    best = None; cur = {}                      # cur: giá trị OR -> l lớn nhất (kết thúc tại r)
    for r, x in enumerate(nums):
        nxt = {}
        for v, l in cur.items():
            w = v | x
            if nxt.get(w, -1) < l:
                nxt[w] = l
        nxt[x] = r
        cur = nxt
        for v, l in cur.items():
            cand = (abs(v - k), r - l, l, r)
            if best is None or cand < best:
                best = cand
    return best[2], best[3]

def many_k(nums, ks):
    vals = set(); cur = set()
    for x in nums:
        cur = {v | x for v in cur}; cur.add(x); vals |= cur
    s = sorted(vals); res = []
    for k in ks:
        i = bisect_left(s, k); d = float('inf')
        if i < len(s): d = min(d, s[i] - k)
        if i > 0: d = min(d, k - s[i - 1])
        res.append(d)
    return res

for _ in range(250):
    a = [rng.randint(1, 31) for _ in range(rng.randint(1, 12))]; k = rng.randint(1, 40)
    l, r = shortest_best(a, k)
    c = 0
    for t in range(l, r + 1):
        c |= a[t]
    best = brute(a, k)
    assert abs(c - k) == best
    shortest = min(r2 - l2 for l2 in range(len(a)) for r2 in range(l2, len(a))
                   if abs(__import__('functools').reduce(lambda p, q: p | q, a[l2:r2 + 1]) - k) == best)
    assert r - l == shortest
    ks = [rng.randint(1, 40) for _ in range(5)]
    assert many_k(a, ks) == [brute(a, q) for q in ks]
print('Bài 3 (mảng con ngắn nhất) và Bài 5 (nhiều truy vấn k) khớp brute.')

Bài 3 (mảng con ngắn nhất) và Bài 5 (nhiều truy vấn k) khớp brute.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
b = list(nums); best = inf
for r, x in enumerate(b):
    best = min(best, abs(x - k)); i = r - 1
    while i >= 0 and b[i] | x != b[i]:
        b[i] |= x; best = min(best, abs(b[i] - k)); i -= 1
```

**Checklist:**

- [ ] Xét cả OR `> k` và `≤ k`?
- [ ] Mảng con một phần tử `[x]` được so?
- [ ] Điều kiện dừng ghi đè `b[i] | x == b[i]`?
- [ ] Không "trừ" OR bằng XOR trong cửa sổ trượt?